CONFIGURACIÓN BASE PARA TODOS LOS LABS

In [1]:
import requests

from pyspark.sql import SparkSession


def reset_spark():

    try:
        data = requests.get(
            "http://localhost:8080/json"
        ).json()

        for app in data["activeapps"]:
            requests.post(
                "http://localhost:8080/app/kill/",
                data={"id": app["id"]}
            )
    except:
        pass

    return SparkSession.builder \
        .appName("EthicalAI") \
        .master("spark://spark-master:7077") \
        .config(
            "spark.jars",
            "/opt/extra-jars/hadoop-aws-3.3.4.jar,"
            "/opt/extra-jars/aws-java-sdk-bundle-1.12.262.jar"
        ) \
        .config(
            "spark.driver.extraClassPath",
            "/opt/extra-jars/*"
        ) \
        .config(
            "spark.executor.extraClassPath",
            "/opt/extra-jars/*"
        ) \
        .config(
            "spark.hadoop.fs.s3a.impl",
            "org.apache.hadoop.fs.s3a.S3AFileSystem"
        ) \
        .config(
            "spark.hadoop.fs.s3a.endpoint",
            "http://minio:9000"
        ) \
        .config(
            "spark.hadoop.fs.s3a.access.key",
            "admin"
        ) \
        .config(
            "spark.hadoop.fs.s3a.secret.key",
            "password123"
        ) \
        .config(
            "spark.hadoop.fs.s3a.path.style.access",
            "true"
        ) \
        .getOrCreate()


spark = reset_spark()

26/05/23 02:14:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


CARGA DE DATOS

In [2]:
from pyspark.sql import functions as F


df = spark.read.csv(
    "/workspace/data/german_credit_data.csv",
    header=True,
    inferSchema=True
)


df.show()

26/05/23 02:15:15 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                

+---+---+------+---+-------+---------------+----------------+-------------+--------+-------------------+----+
|_c0|Age|   Sex|Job|Housing|Saving accounts|Checking account|Credit amount|Duration|            Purpose|Risk|
+---+---+------+---+-------+---------------+----------------+-------------+--------+-------------------+----+
|  0| 67|  male|  2|    own|             NA|          little|         1169|       6|           radio/TV|good|
|  1| 22|female|  2|    own|         little|        moderate|         5951|      48|           radio/TV| bad|
|  2| 49|  male|  1|    own|         little|              NA|         2096|      12|          education|good|
|  3| 45|  male|  2|   free|         little|          little|         7882|      42|furniture/equipment|good|
|  4| 53|  male|  2|   free|         little|          little|         4870|      24|                car| bad|
|  5| 35|  male|  1|   free|             NA|              NA|         9055|      36|          education|good|
|  6| 53| 

PREPROCESAMIENTO BASE

In [4]:
from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline


categorical_cols = [
    "Sex",
    "Housing",
    "Saving accounts",
    "Checking account",
    "Purpose"
]

indexers = [
    StringIndexer(
        inputCol=col,
        outputCol=f"{col}Index",
        handleInvalid="keep"
    )
    for col in categorical_cols
]


feature_columns = [
    "Age",
    "Credit amount",
    "Duration"
] + [f"{c}Index" for c in categorical_cols]


assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features",
    handleInvalid="skip"
)


lr = LogisticRegression(
    featuresCol="features",
    labelCol="Risk"
)


pipeline = Pipeline(
    stages=indexers + [assembler, lr]
)


train, test = df.randomSplit([0.8, 0.2], seed=42)


model = pipeline.fit(train)


predictions = model.transform(test)

IllegalArgumentException: Risk does not exist. Available: _c0, Age, Sex, Job, Housing, Saving accounts, Checking account, Credit amount, Duration, Purpose, SexIndex, HousingIndex, Saving accountsIndex, Checking accountIndex, PurposeIndex, features